# Dullahan practical capability and load benchmark

This notebook runs **real CPU inference** through `dullahan-inference`, accepts your own queries and direct prompts, exercises the full planner → CAL → EDL → expert path, and records reproducible latency, throughput, resource, routing, and semantic-coverage measurements.

Install the notebook environment once from the repository root with `uv sync --extra notebook`, ensure Ollama is running, and install the configured model (default: `ollama pull qwen3:8b`). A single laptop run is a local capacity test, not proof of distributed production scale; increase concurrency and repetitions gradually while watching memory pressure.

In [ ]:
from __future__ import annotations

import atexit
import json
import os
import shutil
import socket
import subprocess
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import UTC, datetime
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import pandas as pd
import psutil
import yaml

candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((path.resolve() for path in candidates if (path / 'pyproject.toml').exists()), None)
if REPO_ROOT is None:
    raise RuntimeError('Run this notebook from the repository root or notebooks/ directory.')

SOURCE_ROOTS = [
    REPO_ROOT / 'apps/agent-runtime/src',
    REPO_ROOT / 'apps/cal/src',
    REPO_ROOT / 'apps/edl/src',
    REPO_ROOT / 'apps/inference/src',
    REPO_ROOT / 'packages/shared/src',
    REPO_ROOT / 'packages/kg/src',
    REPO_ROOT / 'packages/world-state/src',
]
for source_root in reversed(SOURCE_ROOTS):
    if str(source_root) not in sys.path:
        sys.path.insert(0, str(source_root))

from agent_runtime.agent import AgentRuntime
from agent_runtime.config import AgentRuntimeConfig
from agent_runtime.models import AgentRunRequest
from dullahan_inference.config import InferenceConfig
from dullahan_shared.schemas.execution import ExecutionLimits

print(f'Repository: {REPO_ROOT}')
print(f'Python: {sys.executable}')

## 1. Workload configuration

Type a query and/or direct model prompt below. For a scored custom query, add the concepts a useful answer should contain. Concurrency controls simultaneous root Dullahan runs; each root run may also fan out to multiple experts.

In [ ]:
MODEL = 'qwen3:8b'
EMBEDDING_MODEL = 'qwen3-embedding:0.6b'
CUSTOM_QUERY = ''  # Example: 'Create a rollback-safe migration plan for our billing database.'
CUSTOM_REQUIRED_TERMS = []  # Example: ['backup', 'dual-write', 'rollback', 'verification']
CUSTOM_PROMPT = ''  # Direct inference baseline; bypasses Dullahan orchestration.

BUILTIN_CASES = [
    {
        'id': 'database-incident',
        'query': (
            'A production PostgreSQL primary is saturating CPU after a release, replication lag is '
            'growing, and checkout latency is breaching SLOs. Produce an ordered incident response '
            'plan that protects data, restores service, and preserves evidence for root-cause analysis.'
        ),
        'required_terms': ['rollback', 'replication', 'traffic', 'evidence'],
    },
    {
        'id': 'payment-api-design',
        'query': (
            'Design a production payment API that is safe against duplicate requests and abuse. '
            'Cover authentication, idempotency, rate limiting, auditability, failure recovery, and observability.'
        ),
        'required_terms': ['authentication', 'idempotency', 'rate', 'audit', 'observability'],
    },
    {
        'id': 'zero-downtime-migration',
        'query': (
            'Plan a zero-downtime migration from local file state to PostgreSQL. Include dual writes, '
            'backfill, checksum verification, read cutover, rollback criteria, and cleanup gates.'
        ),
        'required_terms': ['dual', 'backfill', 'checksum', 'cutover', 'rollback'],
    },
]

DIRECT_PROMPTS = [
    'In five concise steps, explain how to triage a sudden database latency regression.',
]
if CUSTOM_PROMPT.strip():
    DIRECT_PROMPTS.append(CUSTOM_PROMPT.strip())

CASES = list(BUILTIN_CASES)
if CUSTOM_QUERY.strip():
    CASES.append({
        'id': 'custom',
        'query': CUSTOM_QUERY.strip(),
        'required_terms': [term.lower() for term in CUSTOM_REQUIRED_TERMS],
    })

SMOKE_MODE = os.getenv('DULLAHAN_NOTEBOOK_SMOKE') == '1'
if SMOKE_MODE:
    CASES = CASES[:1]
    DIRECT_PROMPTS = DIRECT_PROMPTS[:1]

CONCURRENCY_LEVELS = [1] if SMOKE_MODE else [1, 2]
REPETITIONS = 1
MAX_DEPTH = 1
MAX_BREADTH = 2
MAX_TOTAL_INSTANCES = 8
EXPERT_MAX_TOKENS = 96 if SMOKE_MODE else 192
REQUEST_TIMEOUT_SECONDS = 240
SAVE_RESULTS = not SMOKE_MODE and os.getenv('DULLAHAN_NOTEBOOK_NO_SAVE') != '1'
MANAGE_INFERENCE_SERVER = True
EXTERNAL_INFERENCE_BASE_URL = 'http://127.0.0.1:30000/v1'

pd.DataFrame(CASES)[['id', 'query', 'required_terms']]

## 2. Start real CPU inference

The managed mode starts Dullahan's OpenAI-compatible Ollama proxy on a free localhost port and forces CPU placement for generation, semantic embeddings, and native tokenizer accounting. Set `MANAGE_INFERENCE_SERVER = False` to benchmark an already-running Dullahan inference endpoint.

In [ ]:
def free_port() -> int:
    with socket.socket() as listener:
        listener.bind(('127.0.0.1', 0))
        return int(listener.getsockname()[1])


def wait_for_health(url: str, process: subprocess.Popen | None, timeout: float = 30) -> dict:
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process is not None and process.poll() is not None:
            raise RuntimeError('dullahan-inference exited before becoming healthy.')
        try:
            with urlopen(url, timeout=1) as response:
                return json.loads(response.read())
        except URLError:
            time.sleep(0.2)
    raise TimeoutError(f'Inference server did not become healthy at {url}')


inference_process = None
inference_tempdir = None
inference_log = None


def stop_inference() -> None:
    global inference_process, inference_tempdir, inference_log
    if inference_process is not None and inference_process.poll() is None:
        inference_process.terminate()
        try:
            inference_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            inference_process.kill()
    if inference_log is not None:
        inference_log.close()
    if inference_tempdir is not None:
        inference_tempdir.cleanup()
    inference_process = None


if MANAGE_INFERENCE_SERVER:
    if shutil.which('ollama') is None:
        raise RuntimeError('Ollama is required. Install it and run: ollama pull ' + MODEL)
    installed = subprocess.run(['ollama', 'list'], check=True, text=True, capture_output=True).stdout
    if MODEL.split(':')[0] not in installed:
        raise RuntimeError(f'{MODEL} is not installed. Run: ollama pull {MODEL}')
    if EMBEDDING_MODEL.split(':')[0] not in installed:
        raise RuntimeError(
            f'{EMBEDDING_MODEL} is not installed. Run: ollama pull {EMBEDDING_MODEL}'
        )

    port = free_port()
    inference_tempdir = tempfile.TemporaryDirectory(prefix='dullahan-notebook-')
    config_path = Path(inference_tempdir.name) / 'inference.yaml'
    inference_config = InferenceConfig(
        provider='ollama',
        device='cpu',
        ollama={
            'model': MODEL,
            'request_timeout_seconds': REQUEST_TIMEOUT_SECONDS,
            'think': False,
        },
        embeddings={
            'model': EMBEDDING_MODEL,
            'dimensions': 1024,
        },
        server={
            'host': '127.0.0.1',
            'advertised_host': '127.0.0.1',
            'port': port,
            'log_level': 'warning',
        },
    )
    config_path.write_text(
        yaml.safe_dump(inference_config.model_dump(mode='json'), sort_keys=False),
        encoding='utf-8',
    )
    environment = os.environ.copy()
    environment['PYTHONPATH'] = os.pathsep.join(
        [*(str(path) for path in SOURCE_ROOTS), environment.get('PYTHONPATH', '')]
    ).rstrip(os.pathsep)
    inference_log = open(Path(inference_tempdir.name) / 'inference.log', 'w+', encoding='utf-8')
    inference_process = subprocess.Popen(
        [sys.executable, '-m', 'dullahan_inference.cli', 'serve', '--config', str(config_path)],
        stdout=inference_log,
        stderr=subprocess.STDOUT,
        text=True,
        env=environment,
    )
    INFERENCE_BASE_URL = f'http://127.0.0.1:{port}/v1'
    health = wait_for_health(f'http://127.0.0.1:{port}/health', inference_process)
else:
    INFERENCE_BASE_URL = EXTERNAL_INFERENCE_BASE_URL.rstrip('/')
    health = wait_for_health(INFERENCE_BASE_URL.removesuffix('/v1') + '/health', None)

atexit.register(stop_inference)
print(f'Inference endpoint: {INFERENCE_BASE_URL}')
print(f"Engine: {health['plan']['engine']} | Device: {health['plan']['device']} | Model: {health['plan']['model']}")

## 3. Direct inference baseline

This isolates raw model latency and token generation from Dullahan orchestration.

In [ ]:
def complete(prompt: str, max_tokens: int = 128) -> dict:
    request = Request(
        f'{INFERENCE_BASE_URL}/completions',
        data=json.dumps({
            'model': 'local-planner',
            'prompt': prompt,
            'max_tokens': max_tokens,
            'temperature': 0,
        }).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    started = time.perf_counter()
    with urlopen(request, timeout=REQUEST_TIMEOUT_SECONDS) as response:
        payload = json.loads(response.read())
    latency = time.perf_counter() - started
    usage = payload.get('usage', {})
    return {
        'prompt': prompt,
        'response': payload['choices'][0]['text'],
        'latency_seconds': latency,
        'prompt_tokens': usage.get('prompt_tokens', 0),
        'completion_tokens': usage.get('completion_tokens', 0),
        'tokens_per_second': usage.get('completion_tokens', 0) / latency if latency else 0,
    }


direct_records = [complete(prompt, max_tokens=96 if SMOKE_MODE else 128) for prompt in DIRECT_PROMPTS]
direct_df = pd.DataFrame(direct_records)
display(direct_df[['latency_seconds', 'completion_tokens', 'tokens_per_second', 'response']])

## 4. Full Dullahan practical and load benchmark

Semantic coverage is transparent and deterministic: it is the fraction of each case's required concepts found in the combined expert responses. It is useful for regression tracking, but it is not a substitute for expert review.

In [ ]:
os.environ['EDL_MODEL_PROVIDER'] = 'http'
os.environ['EDL_MODEL_BASE_URL'] = INFERENCE_BASE_URL
os.environ['EDL_MODEL_TIMEOUT_SECONDS'] = str(REQUEST_TIMEOUT_SECONDS)
os.environ['EDL_MODEL_MAX_TOKENS'] = str(EXPERT_MAX_TOKENS)
os.environ['DULLAHAN_INFERENCE_BASE_URL'] = INFERENCE_BASE_URL
os.environ['DULLAHAN_EMBEDDING_MODEL'] = EMBEDDING_MODEL
os.environ['DULLAHAN_EMBEDDING_DIMENSIONS'] = '1024'
os.environ['DULLAHAN_TOKENIZER_MODEL'] = 'Qwen/Qwen3-8B'
os.environ['DULLAHAN_INFERENCE_TIMEOUT_SECONDS'] = str(REQUEST_TIMEOUT_SECONDS)


def resource_snapshot() -> dict:
    ollama_rss = 0
    for process in psutil.process_iter(['name', 'cmdline', 'memory_info']):
        try:
            identity = ' '.join([process.info.get('name') or '', *(process.info.get('cmdline') or [])]).lower()
            if 'ollama' in identity:
                ollama_rss += process.info['memory_info'].rss
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            continue
    memory = psutil.virtual_memory()
    return {'system_used_bytes': memory.used, 'ollama_rss_bytes': ollama_rss}


def build_runtime() -> AgentRuntime:
    config = AgentRuntimeConfig(
        repo_root=REPO_ROOT,
        limits=ExecutionLimits(
            max_depth=MAX_DEPTH,
            max_breadth_per_agent=MAX_BREADTH,
            max_total_agent_instances=MAX_TOTAL_INSTANCES,
            timeout_seconds_per_instance=REQUEST_TIMEOUT_SECONDS,
        ),
        planner_provider='http',
        planner_model='local-planner',
        planner_base_url=INFERENCE_BASE_URL,
        planner_timeout_seconds=REQUEST_TIMEOUT_SECONDS,
        synthesis_provider='http',
        synthesis_model='local-planner',
        synthesis_base_url=INFERENCE_BASE_URL,
        synthesis_timeout_seconds=REQUEST_TIMEOUT_SECONDS,
        synthesis_max_tokens=EXPERT_MAX_TOKENS,
    )
    return AgentRuntime.local(config)


def run_case(case: dict, concurrency: int, repetition: int) -> dict:
    before = resource_snapshot()
    started = time.perf_counter()
    try:
        result = build_runtime().run(AgentRunRequest(query=case['query']))
        latency = time.perf_counter() - started
        final_response = result.final_response
        normalized = final_response.lower()
        required = [term.lower() for term in case.get('required_terms', [])]
        matched = [term for term in required if term in normalized]
        coverage = len(matched) / len(required) if required else float('nan')
        expert_completion_tokens = sum(
            int(response.routing_metadata.get('model_token_count', 0))
            for response in result.expert_responses
        )
        synthesis_span = next(
            (span for span in result.spans if span.name == 'agent.synthesis'), None
        )
        synthesis_completion_tokens = (
            int(synthesis_span.attributes.get('completion_token_count', 0))
            if synthesis_span else 0
        )
        completion_tokens = expert_completion_tokens + synthesis_completion_tokens
        expert_ids = sorted({response.expert_id for response in result.expert_responses})
        success = bool(result.expert_responses) and bool(final_response.strip())
        error = ''
        subquery_count = len(result.subqueries)
        response_count = len(result.expert_responses)
        mean_confidence = (
            sum(response.confidence or 0 for response in result.expert_responses) / response_count
            if response_count else 0
        )
    except Exception as exc:
        latency = time.perf_counter() - started
        coverage = 0.0
        matched = []
        completion_tokens = 0
        synthesis_completion_tokens = 0
        expert_ids = []
        success = False
        error = f'{type(exc).__name__}: {exc}'
        final_response = ''
        subquery_count = 0
        response_count = 0
        mean_confidence = 0
    after = resource_snapshot()
    return {
        'case_id': case['id'],
        'query': case['query'],
        'concurrency': concurrency,
        'repetition': repetition,
        'success': success,
        'latency_seconds': latency,
        'subquery_count': subquery_count,
        'expert_response_count': response_count,
        'completion_tokens': completion_tokens,
        'synthesis_completion_tokens': synthesis_completion_tokens,
        'semantic_coverage': coverage,
        'matched_terms': matched,
        'expert_ids': expert_ids,
        'mean_routing_confidence': mean_confidence,
        'system_used_delta_mb': (after['system_used_bytes'] - before['system_used_bytes']) / 1_000_000,
        'ollama_rss_delta_mb': (after['ollama_rss_bytes'] - before['ollama_rss_bytes']) / 1_000_000,
        'final_response': final_response,
        'error': error,
    }


records = []
load_records = []
for concurrency in CONCURRENCY_LEVELS:
    workload = [(case, repetition) for repetition in range(REPETITIONS) for case in CASES]
    wave_started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = [executor.submit(run_case, case, concurrency, repetition) for case, repetition in workload]
        for future in as_completed(futures):
            records.append(future.result())
    wave_seconds = time.perf_counter() - wave_started
    wave = [record for record in records if record['concurrency'] == concurrency]
    load_records.append({
        'concurrency': concurrency,
        'runs': len(wave),
        'wall_seconds': wave_seconds,
        'runs_per_minute': 60 * len(wave) / wave_seconds if wave_seconds else 0,
        'success_rate': sum(record['success'] for record in wave) / len(wave) if wave else 0,
    })

results_df = pd.DataFrame(records).sort_values(['concurrency', 'case_id', 'repetition'])
load_df = pd.DataFrame(load_records)
display(results_df[[
    'case_id', 'concurrency', 'success', 'latency_seconds', 'subquery_count',
    'expert_response_count', 'completion_tokens', 'semantic_coverage',
    'mean_routing_confidence', 'error',
]])
display(load_df)
if SMOKE_MODE and (results_df['error'].astype(bool).any() or not results_df['success'].all()):
    failures = results_df.loc[~results_df['success'], ['case_id', 'error']].to_dict('records')
    raise RuntimeError(f'Notebook smoke benchmark failed: {failures}')

## 5. Benchmark visualizations

In [ ]:
plot_df = results_df.copy()
plot_df['semantic_coverage_percent'] = 100 * plot_df['semantic_coverage']

latency = plot_df.groupby('concurrency')['latency_seconds'].agg(
    p50='median', p95=lambda values: values.quantile(0.95)
).reset_index()
coverage = plot_df.groupby('case_id', as_index=False)['semantic_coverage_percent'].mean()
resources = plot_df.groupby('concurrency', as_index=False).agg(
    system_memory_delta_mb=('system_used_delta_mb', 'max'),
    ollama_rss_delta_mb=('ollama_rss_delta_mb', 'max'),
)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

axes[0, 0].plot(latency['concurrency'], latency['p50'], marker='o', label='p50')
axes[0, 0].plot(latency['concurrency'], latency['p95'], marker='s', label='p95')
axes[0, 0].set(title='End-to-end latency under load', xlabel='Concurrent root runs', ylabel='Seconds')
axes[0, 0].set_xticks(latency['concurrency'])
axes[0, 0].grid(alpha=0.25)
axes[0, 0].legend()

axes[0, 1].plot(load_df['concurrency'], load_df['runs_per_minute'], marker='o')
axes[0, 1].set(title='Observed throughput', xlabel='Concurrent root runs', ylabel='Completed runs / minute')
axes[0, 1].set_xticks(load_df['concurrency'])
axes[0, 1].grid(alpha=0.25)

bars = axes[1, 0].bar(coverage['case_id'], coverage['semantic_coverage_percent'])
axes[1, 0].set(title='Required-concept coverage', xlabel='Practical scenario', ylabel='Coverage (%)', ylim=(0, 100))
axes[1, 0].tick_params(axis='x', rotation=20)
axes[1, 0].bar_label(bars, fmt='%.0f%%')

width = 0.35
x = list(range(len(resources)))
axes[1, 1].bar([value - width / 2 for value in x], resources['system_memory_delta_mb'], width, label='System used')
axes[1, 1].bar([value + width / 2 for value in x], resources['ollama_rss_delta_mb'], width, label='Ollama RSS')
axes[1, 1].set(title='Peak observed memory deltas', xlabel='Concurrent root runs', ylabel='MB')
axes[1, 1].set_xticks(x, resources['concurrency'])
axes[1, 1].axhline(0, color='grey', linewidth=0.8)
axes[1, 1].legend()

fig.suptitle(f'Dullahan practical benchmark — {MODEL} on CPU')
display(fig)
plt.close(fig)

## 6. Inspect answers and export raw evidence

Review complete responses before treating a semantic score as meaningful. Raw outputs make regressions auditable and allow later human or independent-model evaluation.

In [ ]:
for record in records:
    print('=' * 100)
    print(f"{record['case_id']} | concurrency={record['concurrency']} | latency={record['latency_seconds']:.2f}s")
    print(f"Matched required terms: {record['matched_terms']}")
    if record['error']:
        print('ERROR:', record['error'])
    else:
        print(record['final_response'])

if SAVE_RESULTS:
    timestamp = datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')
    output_dir = REPO_ROOT / 'notebooks' / 'results' / timestamp
    output_dir.mkdir(parents=True, exist_ok=True)
    results_df.to_csv(output_dir / 'agent-runs.csv', index=False)
    load_df.to_csv(output_dir / 'load-summary.csv', index=False)
    direct_df.to_csv(output_dir / 'direct-inference.csv', index=False)
    (output_dir / 'configuration.json').write_text(json.dumps({
        'model': MODEL,
        'concurrency_levels': CONCURRENCY_LEVELS,
        'repetitions': REPETITIONS,
        'max_depth': MAX_DEPTH,
        'max_breadth': MAX_BREADTH,
        'expert_max_tokens': EXPERT_MAX_TOKENS,
        'cases': CASES,
    }, indent=2), encoding='utf-8')
    print(f'Wrote benchmark evidence to {output_dir}')

In [ ]:
stop_inference()
print('Managed inference server stopped.')

## Reading the results

- Rising throughput with acceptable p95 latency indicates useful local concurrency headroom.
- Falling semantic coverage under load may indicate truncation, timeouts, or model-quality degradation.
- Negative memory deltas can occur because sampling is point-in-time and the OS/Ollama may release memory between snapshots.
- Before a production decision, add domain-specific cases, run multiple repetitions, compare against a baseline system, and have humans or an independent evaluator grade correctness, safety, and completeness.